In [1]:
import pandas as pd
import numpy as np
data_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset.csv"
df = pd.read_csv(data_path)
df['datetime'] = pd.to_datetime(df['datetime'])

print('Dataset loaded successfully')
print(f'Shape: {df.shape}')
print(df.columns.tolist())

Dataset loaded successfully
Shape: (40976, 15)
['datetime', 'city', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month']


In [2]:
df['hour'] = df['datetime'].dt.hour
df['is_smog_prone_hour'] = df['hour'].apply(lambda h: 1 if 4<= h <=9 else 0)
print(df[['datetime', 'hour', 'is_smog_prone_hour']].head(10))
print(f"\nTotal smog-prone hour readings:{df['is_smog_prone_hour'].sum()} out of {len(df)}")

             datetime  hour  is_smog_prone_hour
0 2021-11-01 00:00:00     0                   0
1 2021-11-01 01:00:00     1                   0
2 2021-11-01 02:00:00     2                   0
3 2021-11-01 03:00:00     3                   0
4 2021-11-01 04:00:00     4                   1
5 2021-11-01 05:00:00     5                   1
6 2021-11-01 06:00:00     6                   1
7 2021-11-01 07:00:00     7                   1
8 2021-11-01 08:00:00     8                   1
9 2021-11-01 09:00:00     9                   1

Total smog-prone hour readings:10195 out of 40976


In [3]:
df['dew_point_depression'] = df['temp']-df['dew']

print(df[['temp', 'dew','dew_point_depression']].describe())
print(df[['temp', 'dew','dew_point_depression']].head(10))

              temp           dew  dew_point_depression
count  40976.00000  40976.000000          40976.000000
mean      14.54939      7.983027              6.566363
std        5.96970      5.141336              5.067259
min       -1.20000    -22.000000            -21.500000
25%       10.00000      5.300000              2.700000
50%       14.00000      8.300000              5.000000
75%       19.00000     11.000000              9.700000
max       34.50000     28.300000             38.100000
   temp   dew  dew_point_depression
0  19.0  12.0                   7.0
1  19.0  12.0                   7.0
2  18.5  12.3                   6.2
3  16.9  11.1                   5.8
4  17.0  12.0                   5.0
5  16.5  10.9                   5.6
6  16.0  11.1                   4.9
7  16.0  11.1                   4.9
8  17.0  11.5                   5.5
9  18.0  11.3                   6.7


In [4]:
df = pd.get_dummies(df, columns =['city'], prefix = 'city')
print(df.columns.tolist())
print(df[[c for c in df.columns if c.startswith('city_')]].head())

['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan']
   city_Faisalabad  city_Islamabad  city_Lahore  city_Multan
0            False           False         True        False
1            False           False         True        False
2            False           False         True        False
3            False           False         True        False
4            False           False         True        False


In [5]:
if 'dew_point_depressiom' in df.columns:
    df = df.drop(columns=['dew_point_depressiom'])

print(df.columns.tolist())
print(f"Shape: {df.shape}")

['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan']
Shape: (40976, 21)


In [6]:
city_cols = ['city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan']
df['city_temp'] = df[city_cols].idxmax(axis=1).replace('city_', '', regex=False)
df = df.sort_values(['city_temp','datetime']).reset_index(drop=True)

df['visibility_prev_hour'] = df.groupby('city_temp')['visibility'].shift(1)
print(df[['city_temp','datetime','visibility','visibility_prev_hour']].head(10))

print(f"\nMissing values in new column:{df['visibility_prev_hour'].isnull().sum()}")

df = df.drop(columns=['city_temp'])
print(f"\nFinal shape: {df.shape}")

         city_temp            datetime  visibility  visibility_prev_hour
0  city_Faisalabad 2021-11-01 02:00:00         4.0                   NaN
1  city_Faisalabad 2021-11-01 05:00:00         4.0                   4.0
2  city_Faisalabad 2021-11-01 08:00:00         1.0                   4.0
3  city_Faisalabad 2021-11-01 11:00:00         2.0                   1.0
4  city_Faisalabad 2021-11-01 14:00:00         2.0                   2.0
5  city_Faisalabad 2021-11-01 17:00:00         2.0                   2.0
6  city_Faisalabad 2021-11-01 20:00:00         2.0                   2.0
7  city_Faisalabad 2021-11-01 23:00:00         2.0                   2.0
8  city_Faisalabad 2021-11-02 02:00:00         2.0                   2.0
9  city_Faisalabad 2021-11-02 05:00:00         2.0                   2.0

Missing values in new column:4

Final shape: (40976, 22)


In [7]:
# Reconstruct a temporary 'city' column from the one-hot dummies (needed only for correct grouping)
city_cols = ['city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan']
df['city_temp'] = df[city_cols].idxmax(axis=1).str.replace('city_', '', regex=False)

# Sort by city and datetime so each city's time sequence is continuous
df = df.sort_values(['city_temp', 'datetime']).reset_index(drop=True)

# Previous available reading's visibility (honest name — not always exactly 1 hour before)
df['visibility_prev_reading'] = df.groupby('city_temp')['visibility'].shift(1)

# Time gap (in hours) between current row and the previous reading — shows how "stale" the lag is
df['prev_datetime_temp'] = df.groupby('city_temp')['datetime'].shift(1)
df['time_gap_hours'] = (df['datetime'] - df['prev_datetime_temp']).dt.total_seconds() / 3600

print(df[['city_temp', 'datetime', 'visibility', 'visibility_prev_reading', 'time_gap_hours']].head(10))
print(f"\nMissing values in visibility_prev_reading: {df['visibility_prev_reading'].isnull().sum()}")
print(f"\nTime gap distribution:\n{df['time_gap_hours'].value_counts().head(10)}")

# Drop temporary helper columns — city info already in dummy columns
df = df.drop(columns=['city_temp', 'prev_datetime_temp'])
print(f"\nFinal shape: {df.shape}")

    city_temp            datetime  visibility  visibility_prev_reading  \
0  Faisalabad 2021-11-01 02:00:00         4.0                      NaN   
1  Faisalabad 2021-11-01 05:00:00         4.0                      4.0   
2  Faisalabad 2021-11-01 08:00:00         1.0                      4.0   
3  Faisalabad 2021-11-01 11:00:00         2.0                      1.0   
4  Faisalabad 2021-11-01 14:00:00         2.0                      2.0   
5  Faisalabad 2021-11-01 17:00:00         2.0                      2.0   
6  Faisalabad 2021-11-01 20:00:00         2.0                      2.0   
7  Faisalabad 2021-11-01 23:00:00         2.0                      2.0   
8  Faisalabad 2021-11-02 02:00:00         2.0                      2.0   
9  Faisalabad 2021-11-02 05:00:00         2.0                      2.0   

   time_gap_hours  
0             NaN  
1             3.0  
2             3.0  
3             3.0  
4             3.0  
5             3.0  
6             3.0  
7             3.0  
8    

In [8]:
import numpy as np

# Wind direction is circular (0° = 360°), so raw degrees mislead the model
df['winddir_sin'] = np.sin(np.radians(df['winddir']))
df['winddir_cos'] = np.cos(np.radians(df['winddir']))

print(df[['winddir', 'winddir_sin', 'winddir_cos']].describe())
print(df[['winddir', 'winddir_sin', 'winddir_cos']].head(10))

            winddir   winddir_sin   winddir_cos
count  40976.000000  40976.000000  40976.000000
mean     159.373611     -0.001230      0.470674
std      157.310586      0.582865      0.662383
min        0.000000     -1.000000     -1.000000
25%        0.000000     -0.427358      0.052336
50%      122.650000      0.000000      0.777146
75%      302.000000      0.484810      1.000000
max      639.000000      1.000000      1.000000
   winddir   winddir_sin   winddir_cos
0    360.0 -2.449294e-16  1.000000e+00
1    130.0  7.660444e-01 -6.427876e-01
2     50.0  7.660444e-01  6.427876e-01
3     90.0  1.000000e+00  6.123234e-17
4     90.0  1.000000e+00  6.123234e-17
5     50.0  7.660444e-01  6.427876e-01
6     50.0  7.660444e-01  6.427876e-01
7     50.0  7.660444e-01  6.427876e-01
8     50.0  7.660444e-01  6.427876e-01
9     50.0  7.660444e-01  6.427876e-01


In [9]:
# Fix winddir outliers
df = df[df['winddir'] <= 360]
df['winddir_sin'] = np.sin(np.radians(df['winddir']))
df['winddir_cos'] = np.cos(np.radians(df['winddir']))

# Filter unreliable lag values
df = df[df['time_gap_hours'].isna() | (df['time_gap_hours'] <= 2)]

print(f"Final shape: {df.shape}")
print(f"Winddir max: {df['winddir'].max()}")

Final shape: (37234, 26)
Winddir max: 360.0


In [20]:
# Save final feature-engineered dataset
output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_features.csv"
df.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print(f"Final shape: {df.shape}")
print(f"Columns ({len(df.columns)}):")
print(df.columns.tolist())

Dataset saved successfully!
Final shape: (37234, 26)
Columns (26):
['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan', 'visibility_prev_hour', 'visibility_prev_reading', 'time_gap_hours', 'winddir_sin', 'winddir_cos']


In [22]:
# Drop old duplicate lag column
df = df.drop(columns=['visibility_prev_hour'])

# Save again
output_path = r"C:\Users\Admin\OneDrive\Desktop\FYP_Dataset\master_dataset_features.csv"
df.to_csv(output_path, index=False)

print("Saved!")
print(f"Final shape: {df.shape}")
print(df.columns.tolist())

Saved!
Final shape: (37234, 25)
['datetime', 'temp', 'humidity', 'dew', 'windspeed', 'windgust', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility', 'precip', 'snow', 'snowdepth', 'month', 'hour', 'is_smog_prone_hour', 'dew_point_depression', 'city_Faisalabad', 'city_Islamabad', 'city_Lahore', 'city_Multan', 'visibility_prev_reading', 'time_gap_hours', 'winddir_sin', 'winddir_cos']
